# 01. Environment, layouts and the RL problem

UTS 43008 · Assignment 3 · Project 12 · Dihedrals

Imagine a small board game. You have 200 coins to buy an army. You choose where and when troops
enter a defended base. Troops then move and fight using fixed rules. The learner is the commander,
not the individual soldier.

This notebook connects that picture to the actual observations, actions and rewards. It uses
training layouts only. Vaibhav's Stage 5 simulator is the source; changes are recorded in
[Decisions](../docs/DECISIONS.md). The original ZIP is preserved unchanged.

In [ ]:
from pathlib import Path
import os
import sys
import json
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidate = Path(os.environ.get("BASE_SIEGE_ROOT", Path.cwd())).expanduser()
ROOT = next((p for p in [candidate, *candidate.parents] if (p / "env/siege_env.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the complete project, or set BASE_SIEGE_ROOT to its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from env import config as C
from env.siege_env import SiegeEnv, describe_action, deployment_action
from env.render import draw_snapshot
from env.generator import generate_base
from experiments.provenance import runtime_manifest, sha256
RUNS = Path(os.environ.get("BASE_SIEGE_RUNS_DIR", ROOT / "runs")).expanduser().resolve()
OUTPUT = ROOT / "artifacts" / "notebook-sessions" / uuid.uuid4().hex[:12]
OUTPUT.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 12, "axes.labelsize": 10, "font.size": 10})
def save_figure(name):
    plt.savefig(OUTPUT / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()
print(f"Project: {ROOT.name} | Python: {sys.version.split()[0]}")
print(f"This notebook's outputs: {OUTPUT}")

## 1. The learning problem

| RL term | Meaning here |
|---|---|
| Environment | Layout, army, movement, combat and episode rules |
| Agent | Commander choosing purchases and deployment |
| Observation | Eight board layers and nine summary values |
| Action | Buy, finish shopping, deploy or wait |
| Reward | New destruction and milestones, minus army cost |
| Episode | Shopping followed by one battle |
| Policy | A rule mapping observations to action probabilities |

Training means repeatedly playing, measuring rewards and adjusting the policy. A deterministic
simulator can produce varied experience because layouts, actions and augmented views are sampled.

In [ ]:
troops = pd.DataFrame(C.TROOP_STATS).T
display(troops[["hp", "damage_per_tick", "range", "speed", "cost", "target_rule"]])
display(pd.DataFrame(C.BUILDING_STATS).T)
print("Board:", C.GRID_SIZE, "by", C.GRID_SIZE)
print("Budget:", C.RESOURCE_BUDGET, "| Army limit:", C.MAX_ARMY_SIZE)

## 2. Why four troop types?

The tank absorbs damage and prefers defences. Melee troops are cheap attackers. Ranged troops
can attack from four tiles away, including across walls. Breakers target walls and are consumed
by their attack. There is no manual steering after deployment.

Movement uses four-neighbour weighted A*. Combat range uses Euclidean distance. Walls slow
movement but do not block shots. These are separate rules, not inconsistent distance formulas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for motif, ax in zip(C.TRAIN_MOTIFS, axes):
    board = SiegeEnv(base_seed=42, motifs=(motif,))
    board.reset(seed=0)
    draw_snapshot(board.snapshot(), ax, style="presentation", show_hud=False)
    ax.set_title(motif.replace("_", " ").title())
    board.close()
plt.tight_layout()
save_figure("training-layout-families")

## 3. Procedural layouts, not a downloaded dataset

A seed and a wall structure determine a base. The same pair recreates the same layout. Building
positions must be unique, inside the allowed region and away from walls and deployment tiles.
The three structures above appear in training and current development, with separate seeds.
Layered-core is reserved for the final stage of the current workflow. Earlier software checks
included it in development; that history is disclosed, not erased. Final-layout seeds remain protected.

Generation creates candidates. Some fail additional structural requirements, such as Town Hall
coverage. The table records those rejections instead of assuming every candidate is accepted.

In [ ]:
from experiments.pools import validate_structure
layout_rows = []
for seed in range(20):
    for motif in C.TRAIN_MOTIFS:
        spec = generate_base(seed, motif)
        valid, reason = validate_structure(spec)
        layout_rows.append({"seed": seed, "motif": motif, "walls": len(spec.walls),
                            "buildings": len(spec.buildings), "valid": valid, "reason": reason})
layouts = pd.DataFrame(layout_rows)
display(layouts.groupby("motif").agg(candidates=("seed", "size"), valid=("valid", "sum"),
                                     walls=("walls", "mean"), buildings=("buildings", "mean")))
display(layouts.loc[~layouts.valid].groupby(["motif", "reason"]).size().rename("rejected_candidates"))
assert len(layouts) == 20 * len(C.TRAIN_MOTIFS)

## 4. What the policy sees

The spatial input has shape (8, 16, 16). Five layers locate building types; the remaining layers
show building health, defence coverage and troop density. The vector contains four reserve counts,
remaining resources, remaining time, battle phase, destruction and living troops.

Inputs are float32 values in [0, 1]. Normalisation gives quantities comparable numerical scales.
The policy does not receive the renderer's complete internal state.

In [ ]:
env = SiegeEnv(base_seed=42, motifs=("closed_ring",))
obs, info = env.reset(seed=0)
display(pd.DataFrame([{"input": k, "shape": str(v.shape), "dtype": str(v.dtype),
                       "minimum": float(v.min()), "maximum": float(v.max())} for k, v in obs.items()]))
assert env.observation_space.contains(obs)
names = ["Town Hall", "Cannon", "Mortar", "Storage", "Wall", "Health", "Coverage", "Troop density"]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for layer, name, ax in zip(obs["grid"], names, axes.flat):
    ax.imshow(layer, vmin=0, vmax=1, cmap="viridis")
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
save_figure("observation-layers")

In [ ]:
vector_names = ["tank reserve", "melee reserve", "ranged reserve", "breaker reserve",
                "budget remaining", "time remaining", "battle phase", "destruction", "troops alive"]
display(pd.DataFrame({"field": vector_names, "normalised_value": obs["vector"]}))

## 5. Is this fully observable?

No. Troop density does not separately identify each troop's type, health, target or movement
progress. Different internal states can look identical to the policy. Strictly, this is a
partially observed decision problem.

The feed-forward CNN is a compact baseline, without memory of previous observations. A recurrent
policy could be a future experiment; it is not implemented or claimed here.

## 6. Decode the 246 actions

| IDs | Meaning | Count |
|---|---|---:|
| 0–3 | Buy tank, melee, ranged or breaker | 4 |
| 4 | Finish shopping | 1 |
| 5–244 | Four troop types × 60 perimeter positions | 240 |
| 245 | Wait | 1 |

The mask prevents illegal actions, not poor strategy. Deploying where a cannon can hit may be
legal even when it is a bad choice.

In [ ]:
ids = [0, 3, 4, 5, 64, 65, 244, 245]
display(pd.DataFrame({"id": ids, "meaning": [describe_action(a) for a in ids]}))
trace = []
for action in [0, 2, 4, deployment_action("tank", 7), C.ACTION_WAIT]:
    legal_before = int(env.action_masks().sum())
    _, reward, terminated, truncated, step_info = env.step(action)
    trace.append({"action": describe_action(action), "legal_before": legal_before,
        "tick": step_info["tick"], "spent": step_info["spent"], "reward": reward,
        "terminated": terminated, "truncated": truncated})
display(pd.DataFrame(trace))
env.close()

## 7. A worked reward example

Reward is not the same as destruction. Destroyed non-wall value contributes its fraction of 155
total value. Crossing 50% destruction, destroying the Town Hall and clearing the base each add
0.25 once. Army cost contributes −0.30 × spent / 200, charged once as battle begins.

Destroying walls gives no direct reward, but can help troops reach valuable targets. Gamma is 1.0,
so rewards within this finite episode are not discounted.

In [ ]:
example = pd.DataFrame([
    {"component": "Destroyed value: 80 of 155", "reward": 80 / C.TOTAL_BASE_VALUE},
    {"component": "Half-destruction milestone", "reward": C.MILESTONE_HALF_DESTRUCTION},
    {"component": "Other milestones not achieved", "reward": 0.0},
    {"component": "Army cost: 100 resources", "reward": -C.COST_PENALTY_WEIGHT * 100 / C.RESOURCE_BUDGET},
])
display(example)
print(f"Illustrative total return: {example.reward.sum():.4f}")
print(f"Full clear with 200 spent: {1 + .25*3 - .30:.2f}")

## 8. Episode endings

A cleared base, battle deadline or exhausted army is a normal terminal outcome. The 300-tick
deadline belongs to the task, so it is termination. The 70-decision safety guard is truncation;
value bootstrapping treats it differently.

Battle decisions advance up to 10 ticks, stopping early if the battle ends. Shopping advances
zero combat ticks. PPO transitions and combat ticks are therefore not interchangeable.

In [ ]:
empty = SiegeEnv(base_seed=42)
empty.reset(seed=0)
_, reward, terminated, truncated, end = empty.step(C.ACTION_FINISH_SHOPPING)
display(pd.Series({"return": reward, "terminated": terminated, "truncated": truncated,
                   "finish_reason": end["finish_reason"], "ticks": end["tick"]}))
assert terminated and not truncated and end["tick"] == 0
empty.close()

## 9. Follow the source

1. env/generator.py constructs a seeded base.
2. env/siege_env.py implements reset, masks, step, observation and reward.
3. env/combat.py moves troops, chooses attacks, applies damage and checks endings.
4. env/pathfinding.py estimates travel costs including walls.

Explain why a wall can be strategically important when wall reward is zero.